In [2]:
import pandas as pd

In [3]:
air_quality = pd.read_csv("hourly_88101_2024.csv", low_memory=False)

# basic structure and inferred data types
print(f"Shape: {air_quality.shape}\n")
air_quality.info()

# summary of each column
column_summary = pd.DataFrame({
    "dtype": air_quality.dtypes.astype(str),
    "missing": air_quality.isna().sum(),
    "unique_values": air_quality.nunique(dropna=False),
    "suggested_type": [
        "category" if dtype == "object" and unique / len(air_quality) < 0.5 # if the unique-total ratio is less than 50%, consider it a category
        else str(dtype)
        for dtype, unique in zip(air_quality.dtypes, air_quality.nunique())
    ]
})

display(column_summary)

# frequency tables for each column
for column in air_quality.columns:
    print(f"\n--- {column} ---")
    display(air_quality[column].value_counts(dropna=False).head(20))

# preview the data
display(air_quality.head())

Shape: (8139551, 24)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8139551 entries, 0 to 8139550
Data columns (total 24 columns):
 #   Column               Dtype  
---  ------               -----  
 0   State Code           int64  
 1   County Code          int64  
 2   Site Num             int64  
 3   Parameter Code       int64  
 4   POC                  int64  
 5   Latitude             float64
 6   Longitude            float64
 7   Datum                object 
 8   Parameter Name       object 
 9   Date Local           object 
 10  Time Local           object 
 11  Date GMT             object 
 12  Time GMT             object 
 13  Sample Measurement   float64
 14  Units of Measure     object 
 15  MDL                  float64
 16  Uncertainty          float64
 17  Qualifier            object 
 18  Method Type          object 
 19  Method Code          int64  
 20  Method Name          object 
 21  State Name           object 
 22  County Name          object 
 23  Date of La

,dtype,missing,unique_values,suggested_type
State Code,int64,0,53,int64
County Code,int64,0,126,int64
Site Num,int64,0,266,int64
Parameter Code,int64,0,1,int64
POC,int64,0,12,int64
Latitude,float64,0,925,float64
Longitude,float64,0,925,float64
Datum,object,0,2,category
Parameter Name,object,0,1,category
Date Local,object,0,366,category



--- State Code ---


State Code
6     1016518
42     465466
48     441848
39     324363
12     260456
18     238984
26     234639
49     232915
17     223128
27     203095
37     202771
8      192188
53     185623
32     181485
47     180847
13     176844
30     166979
25     163263
55     161062
34     157288
Name: count, dtype: int64


--- County Code ---


County Code
3      408918
1      384346
13     283821
35     249795
31     225432
21     185023
19     184756
73     178931
29     160383
37     158802
11     151797
5      151360
25     146190
27     140958
67     139697
9      138396
163    135071
49     134179
101    123862
111    120767
Name: count, dtype: int64

... (41 outputs left out)

County Name
Clark             156505
Jefferson         134478
San Diego         118516
Washington         98118
Salt Lake          95689
Hawaii             93138
Wayne              83561
Maricopa           77453
Harris             75827
Hamilton           68434
Philadelphia       59638
Lake               59357
Montgomery         59108
St. Louis City     58783
Kent               55051
Jackson            54709
Cook               54525
Allegheny          53245
Cuyahoga           52229
Ventura            51871
Name: count, dtype: int64


--- Date of Last Change ---


Date of Last Change
2025-12-18    203095
2024-07-29    145586
2024-12-23    123837
2024-10-01    110101
2025-01-08    100176
2025-03-12     97336
2024-06-19     96306
2025-02-20     92931
2025-03-21     92466
2024-09-23     89803
2024-09-30     83382
2024-07-01     81247
2024-09-26     80776
2025-03-20     79501
2025-11-06     79039
2024-12-09     72527
2024-09-20     72013
2025-02-14     71412
2024-12-17     70372
2024-12-05     69685
Name: count, dtype: int64

,State Code,County Code,Site Num,Parameter Code,POC,Latitude,Longitude,Datum,Parameter Name,Date Local,...,Units of Measure,MDL,Uncertainty,Qualifier,Method Type,Method Code,Method Name,State Name,County Name,Date of Last Change
0,1,3,10,88101,3,30.497478,-87.880258,NAD83,PM2.5 - Local Conditions,2024-01-02,...,Micrograms/cubic meter (LC),5.0,NaN,NaN,FEM,209,Met One BAM-1022 Mass Monitor w/ VSCC or TE-PM...,Alabama,Baldwin,2024-07-19
1,1,3,10,88101,3,30.497478,-87.880258,NAD83,PM2.5 - Local Conditions,2024-01-02,...,Micrograms/cubic meter (LC),5.0,NaN,NaN,FEM,209,Met One BAM-1022 Mass Monitor w/ VSCC or TE-PM...,Alabama,Baldwin,2024-07-19
2,1,3,10,88101,3,30.497478,-87.880258,NAD83,PM2.5 - Local Conditions,2024-01-02,...,Micrograms/cubic meter (LC),5.0,NaN,NaN,FEM,209,Met One BAM-1022 Mass Monitor w/ VSCC or TE-PM...,Alabama,Baldwin,2024-07-19
3,1,3,10,88101,3,30.497478,-87.880258,NAD83,PM2.5 - Local Conditions,2024-01-02,...,Micrograms/cubic meter (LC),5.0,NaN,NaN,FEM,209,Met One BAM-1022 Mass Monitor w/ VSCC or TE-PM...,Alabama,Baldwin,2024-07-19
4,1,3,10,88101,3,30.497478,-87.880258,NAD83,PM2.5 - Local Conditions,2024-01-02,...,Micrograms/cubic meter (LC),5.0,NaN,NaN,FEM,209,Met One BAM-1022 Mass Monitor w/ VSCC or TE-PM...,Alabama,Baldwin,2024-07-19


* Can we use a combination of keys for our primary key?

* Since this is hourly data, we might have unique values by long,lat,date/time

In [8]:
# Identify rows with duplicate longitude, latitude, and hour values
key_columns = ["Longitude", "Latitude", "Date Local", "Time Local", "Method Code"]

missing_columns = [column for column in key_columns if column not in air_quality.columns]
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

duplicate_mask = air_quality.duplicated(subset=key_columns, keep=False)
duplicates = air_quality.loc[duplicate_mask].sort_values(key_columns)

print(f"Duplicate rows found: {len(duplicates)}")
display(duplicates[key_columns])

Duplicate rows found: 778636


,Longitude,Latitude,Date Local,Time Local,Method Code
2165299,-155.913417,19.509778,2024-01-01,00:00,209
2173917,-155.913417,19.509778,2024-01-01,00:00,209
2165300,-155.913417,19.509778,2024-01-01,01:00,209
2173918,-155.913417,19.509778,2024-01-01,01:00,209
2165302,-155.913417,19.509778,2024-01-01,03:00,209
...,...,...,...,...,...
3468700,-71.056061,42.325186,2024-12-31,21:00,636
3460113,-71.056061,42.325186,2024-12-31,22:00,636
3468701,-71.056061,42.325186,2024-12-31,22:00,636
3460114,-71.056061,42.325186,2024-12-31,23:00,636


In [7]:
duplicates

,State Code,County Code,Site Num,Parameter Code,POC,Latitude,Longitude,Datum,Parameter Name,Date Local,...,Units of Measure,MDL,Uncertainty,Qualifier,Method Type,Method Code,Method Name,State Name,County Name,Date of Last Change
2242416,15,3,10,88101,1,21.323745,-158.088613,WGS84,PM2.5 - Local Conditions,2024-01-01,...,Micrograms/cubic meter (LC),0.1,NaN,NaN,FEM,238,Teledyne T640X at 16.67 LPM - Broadband spectr...,Hawaii,Honolulu,2024-06-10
2251082,15,3,10,88101,21,21.323745,-158.088613,WGS84,PM2.5 - Local Conditions,2024-01-01,...,Micrograms/cubic meter (LC),0.1,NaN,NaN,FEM,738,Teledyne T640X at 16.67 LPM (Corrected) - Broa...,Hawaii,Honolulu,2024-06-10
2242417,15,3,10,88101,1,21.323745,-158.088613,WGS84,PM2.5 - Local Conditions,2024-01-01,...,Micrograms/cubic meter (LC),0.1,NaN,NaN,FEM,238,Teledyne T640X at 16.67 LPM - Broadband spectr...,Hawaii,Honolulu,2024-06-10
2251083,15,3,10,88101,21,21.323745,-158.088613,WGS84,PM2.5 - Local Conditions,2024-01-01,...,Micrograms/cubic meter (LC),0.1,NaN,NaN,FEM,738,Teledyne T640X at 16.67 LPM (Corrected) - Broa...,Hawaii,Honolulu,2024-06-10
2242418,15,3,10,88101,1,21.323745,-158.088613,WGS84,PM2.5 - Local Conditions,2024-01-01,...,Micrograms/cubic meter (LC),0.1,NaN,NaN,FEM,238,Teledyne T640X at 16.67 LPM - Broadband spectr...,Hawaii,Honolulu,2024-06-10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3180788,23,3,1011,88101,4,46.682299,-68.016195,WGS84,PM2.5 - Local Conditions,2024-09-16,...,Micrograms/cubic meter (LC),5.0,NaN,NaN,FEM,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,Maine,Aroostook,2024-10-03
3176371,23,3,1011,88101,3,46.682299,-68.016195,WGS84,PM2.5 - Local Conditions,2024-09-16,...,Micrograms/cubic meter (LC),0.1,NaN,NaN,FEM,638,Teledyne T640X at 16.67 LPM w/Network Data Ali...,Maine,Aroostook,2024-10-03
3180789,23,3,1011,88101,4,46.682299,-68.016195,WGS84,PM2.5 - Local Conditions,2024-09-16,...,Micrograms/cubic meter (LC),5.0,NaN,NaN,FEM,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,Maine,Aroostook,2024-10-03
3176372,23,3,1011,88101,3,46.682299,-68.016195,WGS84,PM2.5 - Local Conditions,2024-09-16,...,Micrograms/cubic meter (LC),0.1,NaN,NaN,FEM,638,Teledyne T640X at 16.67 LPM w/Network Data Ali...,Maine,Aroostook,2024-10-03


* Apparently not. So we probably have to generate our own key since we cant find a reliable combination